In [ ]:
# !pip install -U bertopic[all] sentence-transformers umap-learn hdbscan

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import umap, hdbscan

docs = [
    "Transformers are great for NLP.",
    "I tuned a BERT model for question answering.",
    "Topic modeling helps summarize large corpora.",
    "Cycling in the mountains is my favorite hobby.",
    "Road bikes and gravel bikes need different tires.",
]

n = len(docs)
# guardrails for tiny corpora
safe_n_components = max(2, min(5, n - 2))   # ensure n_components + 1 < n
safe_n_neighbors  = max(2, min(10, n - 1))  # must be < n

umap_model = umap.UMAP(
    n_components=safe_n_components,
    n_neighbors=safe_n_neighbors,
    min_dist=0.0,
    metric="cosine",
    init="random",           # avoids spectral init eigenproblem on tiny N
    random_state=42
)

# For tiny corpora, make clusters small so HDBSCAN doesn’t toss everything as -1
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=max(2, n // 2),  # e.g., with n=5 -> 2 or 3
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print(topic_model.get_topic_info())


2025-10-27 19:18:17,650 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-10-27 19:18:17,676 - BERTopic - Embedding - Completed ✓
2025-10-27 19:18:17,678 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-27 19:18:18,343 - BERTopic - Dimensionality - Completed ✓
2025-10-27 19:18:18,344 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-27 19:18:18,356 - BERTopic - Cluster - Completed ✓
2025-10-27 19:18:18,359 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-27 19:18:18,368 - BERTopic - Representation - Completed ✓


   Topic  Count                       Name  \
0      0      3   0_for_topic_question_nlp   
1      1      2  1_bikes_road_my_mountains   

                                      Representation  \
0  [for, topic, question, nlp, modeling, summariz...   
1  [bikes, road, my, mountains, is, the, in, hobb...   

                                 Representative_Docs  
0  [Transformers are great for NLP., Topic modeli...  
1  [Cycling in the mountains is my favorite hobby...  
